# Situational POSG evaluation — Colab workflow

This notebook **evaluates the already-generated S1–S11 Python files**. It does not call an LLM and does not regenerate outputs.

**Hardware:** choose a normal CPU runtime. A GPU provides no benefit. The run launches one bounded subprocess per saved Python file and usually takes under a minute.

## Steps
1. Put the repository in `/content/world-model` (clone it or mount/copy it from Drive).
2. Run the setup cell.
3. Run the evaluation cell. It evaluates all 217 saved files, including repairs.
4. Inspect the summary and detailed CSV.
5. Optionally generate/display the controlled five-model situational figures.


In [ ]:
from pathlib import Path
import os, subprocess

# Choose ONE setup method.
REPO_URL = ""  # e.g. https://github.com/<organization>/world-model.git
REPO_DIR = Path("/content/world-model")

if not REPO_DIR.exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL above, or upload/mount the repository at /content/world-model")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository:", Path.cwd())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## Run the complete situational evaluation

The authoritative entry point is `experiments/evaluate_python_situational.py`. Existing committed results are under `results/python-situational/evaluation`; rerunning overwrites them deterministically.


In [ ]:
subprocess.run(["python", "experiments/evaluate_python_situational.py"], check=True)


## Read the aggregate and detailed results


In [ ]:
import json
import pandas as pd

summary_path = Path("results/python-situational/evaluation/summary.json")
summary = json.loads(summary_path.read_text())
display(summary)

attempts = pd.read_csv("results/python-situational/evaluation/attempt_results.csv")
display(attempts.head())
display(attempts.groupby(["case", "model"])[["lane1_passed", "lane3_checklist_passed"]].mean())


## Controlled five-model situational visualizations

The evaluation directory includes every generation and repair. For a fair five-model plot, the comparison script selects the same `g0_r0` condition for every model and case.


In [ ]:
subprocess.run(["python", "experiments/compare_five_model_outputs.py"], check=True)
subprocess.run(["python", "experiments/plot_five_model_comparison.py"], check=True)

from IPython.display import SVG, display
for figure in [
    "results/cross_model_comparison/figures/situational_model_radar.svg",
    "results/cross_model_comparison/figures/situational_pairwise_heatmap.svg",
    "results/cross_model_comparison/figures/scenario_agreement_bars.svg",
]:
    print(figure)
    display(SVG(filename=figure))


## Download outputs (optional)


In [ ]:
import shutil
archive = shutil.make_archive("/content/situational_evaluation", "zip", REPO_DIR / "results/python-situational/evaluation")
print("Created", archive)
from google.colab import files
files.download(archive)
